# Chạy project 11 ngành trên Google Colab (Nhóm 01)
Colab có sẵn Linux nên **không cần cài Java/winutils trên máy Windows**. Chạy lần lượt từng ô (Shift + Enter).

| Ô | Việc | Thời gian |
|---|---|---|
| 1 | Upload file zip project và giải nén | ~30 giây |
| 2 | Cài Java 17 + thư viện | ~2 phút |
| 3 | Kiểm tra môi trường | ~30 giây |
| 4 | Lấy dữ liệu DNSE (hoặc 4b: upload dữ liệu lấy từ máy bạn) | ~3–6 phút |
| 5 | Pipeline PySpark | ~2–4 phút |
| 6 | Dashboard + insight + kiểm thử | ~1 phút |
| 7 | Tải kết quả về máy | |
| 8 | (Tùy chọn) Mở Streamlit ngay trên Colab | |

## 1. Upload file `Nhom01_DeTaiCK_BigData_11Nganh.zip` và giải nén
Bấm **Choose Files** rồi chọn file zip project.

In [ ]:
from google.colab import files
import os, zipfile

def safe_extract(zip_path, dest):
    """Giải nén, đổi dấu '\\' (zip tạo bằng PowerShell) thành '/' để giữ đúng cấu trúc thư mục."""
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            name = info.filename.replace('\\', '/')
            if name.endswith('/'):
                continue
            out = os.path.join(dest, name)
            os.makedirs(os.path.dirname(out), exist_ok=True)
            with z.open(info) as src, open(out, 'wb') as f:
                f.write(src.read())

up = files.upload()
safe_extract(next(iter(up)), '/content/project_upload')
# tìm thư mục chứa src/spark_pipeline.py (dù zip có thêm lớp thư mục ngoài)
PROJECT = next(r for r, d, f in os.walk('/content/project_upload') if 'spark_pipeline.py' in f).rsplit('/src', 1)[0]
os.chdir(PROJECT)
print('Đang ở thư mục:', os.getcwd())
print(os.listdir())

## 2. Cài Java 17 và thư viện Python

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
!java -version
!pip -q install -r requirements.txt

## 3. Kiểm tra môi trường
Kết quả mong muốn: tất cả `[OK]`. Nếu chỉ mục **5 (API DNSE)** báo lỗi → máy chủ Colab ở nước ngoài bị chặn, dùng **ô 4b** thay cho ô 4.

In [ ]:
!python src/check_env.py

## 4. Lấy dữ liệu từ API DNSE
In ra số phiên của từng mã; dòng cuối cho biết tổng số dòng (mục tiêu > 100.000).

In [ ]:
!python src/fetch_dnse.py

## 4b. (Chỉ khi ô 4 lỗi) Upload dữ liệu thô lấy từ máy bạn
Trên **máy Windows** (chỉ cần Python, không cần Java), mở cmd trong thư mục project và chạy:
```
python -m pip install requests
python src\fetch_dnse.py
powershell Compress-Archive -Path data\raw -DestinationPath raw.zip -Force
```
Sau đó chạy ô dưới và upload file `raw.zip`.

In [ ]:
from google.colab import files
import os
os.chdir(PROJECT)
up = files.upload()
safe_extract(next(iter(up)), os.path.join(PROJECT, 'data'))
# nếu zip chứa thẳng thư mục 'dnse' (không có lớp 'raw') thì chuyển vào data/raw
if os.path.isdir('data/dnse') and not os.path.isdir('data/raw/dnse'):
    os.makedirs('data/raw', exist_ok=True); os.rename('data/dnse', 'data/raw/dnse')
n = sum(len(f) for _, _, f in os.walk('data/raw/dnse/stock'))
print('Số file JSON cổ phiếu:', n, '(cần > 0; khoảng 700-800 nếu đủ 81 mã x 10 năm)')

## 5. Chạy pipeline PySpark (Bronze → Silver → Gold)

In [ ]:
!python src/spark_pipeline.py 2>&1 | grep -v -E "WARN|log4j|Setting default log level|To adjust logging|FutureWarning|require_minimum"

Xem nhanh số dòng qua từng bước và báo cáo chất lượng dữ liệu:

In [ ]:
import json, pandas as pd
m = json.load(open('reports/pipeline_metrics.json', encoding='utf-8'))
display(pd.DataFrame([{'Bước': k, **v} for k, v in m['stages'].items() if k != '4_gold_tables']))
display(pd.read_csv('data/gold/csv/data_quality_report.csv'))
display(pd.read_csv('data/gold/csv/sector_kpi.csv').round(3))

## 6. Dashboard BI + insight + kiểm thử

In [ ]:
!python src/build_dashboard.py
!python -m pytest -q tests 2>&1 | tail -2
from IPython.display import Markdown
Markdown(open('reports/insights_auto.md', encoding='utf-8').read())

## 7. Tải kết quả về máy
File `ket_qua.zip` gồm `data/raw` (dữ liệu gốc DNSE, để tái lập), `data/gold`, `reports`, `dashboard`. Giải nén **vào thư mục project trên máy bạn** (ghi đè), sau đó:
- Mở `dashboard/BI_Dashboard_Nganh_VN.html` bằng trình duyệt.
- Chạy Streamlit trên máy (không cần Java): `python -m pip install streamlit plotly pandas` rồi `python -m streamlit run app\streamlit_app.py`.
- Dùng `data/gold/csv/*.csv` để dựng Power BI (`docs/POWER_BI.md`).

In [ ]:
!cd {PROJECT} && rm -f ket_qua.zip && zip -qr ket_qua.zip data/raw data/gold reports dashboard && ls -lh ket_qua.zip
from google.colab import files
files.download(f'{PROJECT}/ket_qua.zip')

## 8. (Tùy chọn) Mở Streamlit ngay trên Colab
Chạy ô dưới, chờ ~20 giây, bấm vào link dạng `https://....trycloudflare.com` được in ra.
Dừng app: bấm nút ■ (Interrupt) của ô.

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, time, re
subprocess.Popen(['streamlit', 'run', 'app/streamlit_app.py', '--server.port', '8501', '--server.headless', 'true'],
                 stdout=open('streamlit.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    hit = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if hit:
        print('Mở Streamlit tại:', hit.group(0))
        break
tunnel.wait()